# ChurnIQ Model Experiments and Feature Diagnostics\n
\n
This notebook compares a majority-class baseline, balanced Logistic Regression, and Extra Trees on the same stratified holdout set. It then examines churn rates across numeric and categorical features.\n
\n
**Important:** The existing production model is not overwritten. A ROC-AUC close to 0.50 indicates little ranking signal on this split. Group churn rates are descriptive, not causal. The `tenure_months` feature below is calculated relative to the latest registration date in the dataset, so it should not be interpreted as verified current customer tenure without a valid observation date.\n

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the dataset whether this notebook is run from the project root
# or from the notebook folder.
possible_paths = [
    Path("../data/cleaned/telecom_churn_cleaned.csv"),
    Path("data/cleaned/telecom_churn_cleaned.csv"),
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/cleaned/telecom_churn_cleaned.csv. "
        "Run this notebook from the project root or notebook folder."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Original shape:", df.shape)

if "churn" not in df.columns:
    raise ValueError("Expected target column 'churn' was not found.")

df["date_of_registration"] = pd.to_datetime(
    df["date_of_registration"], errors="coerce"
)

# This is a dataset-relative tenure feature, not necessarily real current tenure.
# It is useful for experimentation, but interpret it carefully.
analysis_date = df["date_of_registration"].max()
df["tenure_months"] = (
    (analysis_date - df["date_of_registration"]).dt.days / 30.44
).round(1)
df["registration_year"] = df["date_of_registration"].dt.year

features = [
    "telecom_partner", "gender", "age", "state", "city",
    "num_dependents", "estimated_salary", "calls_made",
    "sms_sent", "data_used", "tenure_months", "registration_year"
]

X = df[features].copy()
y = df["churn"].copy()

print("Feature matrix:", X.shape)
print("Target counts:")
print(y.value_counts(dropna=False).sort_index())
print("Churn rate:", round(y.mean() * 100, 2), "%")


Loaded: C:\Users\Dhruv Srivastava\OneDrive\Desktop\Customer-Churn-Analysis\data\cleaned\telecom_churn_cleaned.csv
Original shape: (243553, 14)
Feature matrix: (243553, 12)
Target counts:
churn
0    194726
1     48827
Name: count, dtype: int64
Churn rate: 20.05 %


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

if y.isna().any():
    raise ValueError("The churn target contains missing values; clean it before training.")

if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected binary churn labels 0 and 1; found {sorted(y.unique())}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = [
    "age", "num_dependents", "estimated_salary",
    "calls_made", "sms_sent", "data_used",
    "tenure_months", "registration_year"
]

categorical_features = [
    "telecom_partner", "gender", "state", "city"
]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training churn rate: {y_train.mean():.2%}")
print(f"Test churn rate: {y_test.mean():.2%}")


Training rows: 194,842
Test rows: 48,711
Training churn rate: 20.05%
Test churn rate: 20.05%


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

models = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),

    "Balanced Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Extra Trees": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", ExtraTreesClassifier(
            n_estimators=150,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(4))

print(
    "\nInterpret ROC-AUC near 0.50 as weak ranking performance. "
    "Do not select a model using accuracy alone when classes are imbalanced."
)


,Accuracy,Balanced Accuracy,Precision,Recall,F1 Score,ROC-AUC
Model,,,,,,
Dummy Baseline,0.7995,0.5000,0.0000,0.0000,0.0000,0.5000
Balanced Logistic Regression,0.5042,0.5013,0.2013,0.4964,0.2864,0.5058
Extra Trees,0.7186,0.5049,0.2117,0.1483,0.1744,0.5042



Interpret ROC-AUC near 0.50 as weak ranking performance. Do not select a model using accuracy alone when classes are imbalanced.


In [4]:
# Diagnostic 1: churn rates across numeric feature groups.
# These are descriptive associations, not causal effects.

numeric_diagnostic_features = [
    "age", "num_dependents", "estimated_salary",
    "calls_made", "sms_sent", "data_used", "tenure_months"
]

numeric_diagnostic = X[numeric_diagnostic_features].copy()
numeric_diagnostic["churn"] = y.to_numpy()

for col in numeric_diagnostic_features:
    valid = numeric_diagnostic[[col, "churn"]].dropna()
    if valid.empty or valid[col].nunique() < 2:
        print(f"\nSkipping {col}: not enough non-missing variation.")
        continue

    valid = valid.copy()
    valid["group"] = pd.qcut(valid[col], q=10, duplicates="drop")

    print(f"\nChurn rate by {col}:")
    display(
        valid.groupby("group", observed=True)["churn"]
        .agg(customers="size", churn_rate="mean")
        .assign(churn_rate_percent=lambda d: d["churn_rate"] * 100)
        .round(2)
    )



Churn rate by age:


,customers,churn_rate,churn_rate_percent
group,,,
"(17.999, 23.0]",25558,0.2,19.98
"(23.0, 29.0]",25401,0.2,19.55
"(29.0, 35.0]",25369,0.2,20.49
"(35.0, 40.0]",21505,0.2,20.18
"(40.0, 46.0]",25392,0.2,20.23
"(46.0, 52.0]",25866,0.2,19.71
"(52.0, 58.0]",25492,0.2,20.13
"(58.0, 63.0]",21797,0.2,20.20
"(63.0, 69.0]",25796,0.2,19.97



Churn rate by num_dependents:


,customers,churn_rate,churn_rate_percent
group,,,
"(-0.001, 1.0]",97628,0.2,20.19
"(1.0, 2.0]",48605,0.2,20.00
"(2.0, 3.0]",48694,0.2,19.81
"(3.0, 4.0]",48626,0.2,20.05



Churn rate by estimated_salary:


...

,customers,churn_rate,churn_rate_percent
group,,,
"(-0.001, 5.0]",24652,0.20,20.12
"(5.0, 10.0]",24303,0.20,20.10
"(10.0, 15.0]",24474,0.20,20.41
"(15.0, 20.0]",24569,0.20,19.86
"(20.0, 25.0]",24235,0.20,19.77
"(25.0, 30.0]",24067,0.20,20.30
"(30.0, 34.0]",19385,0.20,20.15
"(34.0, 39.0]",24219,0.19,19.44
"(39.0, 44.0]",24503,0.20,20.25



Churn rate by data_used:


,customers,churn_rate,churn_rate_percent
group,,,
"(-0.001, 1229.0]",23768,0.2,19.52
"(1229.0, 2189.0]",23741,0.2,19.88
"(2189.0, 3161.0]",23754,0.2,20.02
"(3161.0, 4129.8]",23738,0.2,20.33
"(4129.8, 5112.0]",23762,0.2,20.28
"(5112.0, 6090.0]",23760,0.2,20.01
"(6090.0, 7071.0]",23753,0.2,20.48
"(7071.0, 8047.0]",23731,0.2,20.13
"(8047.0, 9018.0]",23759,0.2,19.76



Churn rate by tenure_months:


,customers,churn_rate,churn_rate_percent
group,,,
"(-0.001, 4.0]",24575,0.2,20.01
"(4.0, 8.0]",24376,0.2,20.14
"(8.0, 12.0]",24175,0.2,19.98
"(12.0, 16.0]",24375,0.2,20.12
"(16.0, 20.0]",24375,0.2,20.36
"(20.0, 24.0]",24376,0.2,19.99
"(24.0, 28.1]",24774,0.2,20.12
"(28.1, 32.1]",24376,0.2,20.01
"(32.1, 36.1]",24375,0.2,19.93


In [5]:
# Diagnostic 2: churn rates across categorical features.
# Small differences should be interpreted cautiously, especially with many groups.

for col in ["telecom_partner", "gender", "state", "city"]:
    print(f"\nChurn rate by {col}:")
    display(
        df.groupby(col, observed=True)["churn"]
        .agg(customers="size", churn_rate="mean")
        .assign(churn_rate_percent=lambda d: d["churn_rate"] * 100)
        .sort_values("churn_rate_percent", ascending=False)
        .round(2)
    )



Churn rate by telecom_partner:


,customers,churn_rate,churn_rate_percent
telecom_partner,,,
Airtel,60905,0.2,20.37
Reliance Jio,61123,0.2,20.02
Vodafone,60802,0.2,19.95
BSNL,60723,0.2,19.86



Churn rate by gender:


,customers,churn_rate,churn_rate_percent
gender,,,
F,97576,0.2,20.30
M,145977,0.2,19.88



Churn rate by state:


,customers,churn_rate,churn_rate_percent
state,,,
Jharkhand,8755,0.21,21.12
Karnataka,8845,0.21,20.71
Mizoram,8689,0.21,20.65
Uttarakhand,8856,0.20,20.42
Himachal Pradesh,8682,0.20,20.41
Madhya Pradesh,8817,0.20,20.38
Assam,8547,0.20,20.30
Odisha,8712,0.20,20.25
Nagaland,8562,0.20,20.23



Churn rate by city:


,customers,churn_rate,churn_rate_percent
city,,,
Hyderabad,40698,0.2,20.42
Mumbai,40386,0.2,20.16
Kolkata,40661,0.2,20.16
Bangalore,40603,0.2,20.05
Chennai,40749,0.2,19.76
Delhi,40456,0.2,19.75


In [6]:

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# Reload the original cleaned data for inspection
df_check = pd.read_csv(DATA_PATH)
y_check = df_check["churn"].astype(int)

print("1. Target distribution")
print(df_check["churn"].value_counts(dropna=False))
print("\nTarget proportions:")
print(df_check["churn"].value_counts(normalize=True).round(4))

print("\n2. Target data type and unique values")
print("Data type:", df_check["churn"].dtype)
print("Unique values:", sorted(y_check.unique()))
print("Missing labels:", df_check["churn"].isna().sum())

print("\n3. Does customer_id uniquely identify each row?")
print("Unique IDs:", df_check["customer_id"].nunique())
print("Total rows:", len(df_check))
print("Duplicate IDs:", df_check["customer_id"].duplicated().sum())

print("\n4. Numeric feature relationships with churn")
numeric_cols = [
    "age", "num_dependents", "estimated_salary",
    "calls_made", "sms_sent", "data_used"
]
numeric_cols = [c for c in numeric_cols if c in df_check.columns]

corr = df_check[numeric_cols].corrwith(y_check).sort_values()
print(corr.round(4))

print("\n5. Churn rate across categories")
for col in ["telecom_partner", "gender", "state", "city"]:
    if col in df_check.columns:
        summary = (
            df_check.groupby(col, dropna=False)["churn"]
            .agg(customers="size", churn_rate="mean")
        )
        summary["churn_rate_percent"] = summary["churn_rate"] * 100
        print(f"\n{col}:")
        print(summary.sort_values("churn_rate_percent").round(2))

print("\n6. Mutual information with churn")
# Mutual information can detect some non-linear relationships.
# This checks numeric features only; it does not prove causation.
X_num = df_check[numeric_cols].copy()
X_num = X_num.fillna(X_num.median())

mi = mutual_info_classif(
    X_num, y_check,
    discrete_features=False,
    random_state=42
)

mi_table = pd.Series(
    mi, index=numeric_cols, name="mutual_information"
).sort_values(ascending=False)

print(mi_table.round(5))

1. Target distribution
churn
0    194726
1     48827
Name: count, dtype: int64

Target proportions:
churn
0    0.7995
1    0.2005
Name: proportion, dtype: float64

2. Target data type and unique values
Data type: int64
Unique values: [0, 1]
Missing labels: 0

3. Does customer_id uniquely identify each row?
Unique IDs: 243553
Total rows: 243553
Duplicate IDs: 0

4. Numeric feature relationships with churn
estimated_salary   -0.0033
num_dependents     -0.0025
sms_sent           -0.0022
age                 0.0008
data_used           0.0016
calls_made          0.0022
dtype: float64

5. Churn rate across categories

telecom_partner:
                 customers  churn_rate  churn_rate_percent
telecom_partner                                           
BSNL                 60723         0.2               19.86
Vodafone             60802         0.2               19.95
Reliance Jio         61123         0.2               20.02
Airtel               60905         0.2               20.37

gender:
 

In [7]:
print("Missing labels:", df_check["churn"].isna().sum())

Missing labels: 0


## Next steps after running\n
\n
- Review whether churn rates change meaningfully across feature groups.\n
- Investigate how the churn target was generated and whether relevant behaviour/history features are absent.\n
- Consider cross-validation and threshold analysis before selecting a model.\n
- Do not replace `models/churn_model.joblib` until a candidate shows a validated improvement and its input schema is compatible with the FastAPI endpoint.\n